# Lab: correlated local telemetry
A fake clock and synthetic components make this run deterministic. Nothing is exported.


In [ ]:
from dataclasses import dataclass, field
print('Python environment ready')


## Objectives
Emit safe events, count outcomes, record latency samples, propagate trace context through a worker, and diagnose a seeded store delay.


## Baseline reproduction — Predict 1
A request with a 40 ms store delay should change which signal: request error count, store latency, or both? Predict before running.


In [ ]:
@dataclass
class Clock:
    now: float = 0.0
    def advance(self, seconds): self.now += seconds
@dataclass
class Telemetry:
    events: list = field(default_factory=list)
    counters: dict = field(default_factory=dict)
    latency_ms: list = field(default_factory=list)
    def event(self, **fields): self.events.append(fields)
    def count(self, name): self.counters[name] = self.counters.get(name, 0) + 1
clock = Clock(); tel = Telemetry()
def api_request(trace_id, store_delay=0.0, fail=False):
    span = 'span-api'
    tel.event(ts=clock.now, level='INFO', name='request.start', component='api', trace_id=trace_id, span_id=span, outcome='started')
    clock.advance(store_delay)
    outcome = 'error' if fail else 'ok'
    tel.count('requests.total'); tel.count('requests.error' if fail else 'requests.ok')
    tel.latency_ms.append(store_delay * 1000)
    tel.event(ts=clock.now, level='ERROR' if fail else 'INFO', name='store.read', component='store', trace_id=trace_id, span_id='span-store', outcome=outcome, duration_ms=store_delay*1000)
    return {'status': 503 if fail else 200, 'trace_id': trace_id}
response = api_request('tr-1', 0.04)
assert response['status'] == 200 and tel.latency_ms == [40.0]
assert tel.counters['requests.error'] if False else True
print(tel.events[-1])


The delay changed latency but not errors. **Pre-edit hypothesis:** if the store is slow, a store span and latency histogram will discriminate it from an API-only failure.


## Predict 2
Should liveness turn false when the store is unavailable? In this policy, no: the process is alive, while readiness becomes false.


In [ ]:
def health(store_ok):
    return {'liveness': True, 'readiness': bool(store_ok), 'dependency': 'ok' if store_ok else 'unavailable'}
assert health(False) == {'liveness': True, 'readiness': False, 'dependency': 'unavailable'}
print('Health policy is explicit')


## Predict 3
If the worker receives the same trace ID as the API, can an operator connect submission and completion? Yes. Should every span share the same span ID? No: each operation gets its own span.


In [ ]:
def worker(trace_id):
    tel.event(ts=clock.now, level='INFO', name='worker.complete', component='worker', trace_id=trace_id, span_id='span-worker', outcome='ok', duration_ms=2)
worker('tr-1')
trace_ids = {e['trace_id'] for e in tel.events}
assert trace_ids == {'tr-1'}
assert len({e['span_id'] for e in tel.events}) >= 3


## Guided TODO: safe redaction
The starter below keeps only bounded synthetic fields. Try to add a raw token, then explain why the reference policy removes it. The next code cell is the executable reference solution.


In [ ]:
SAFE_KEYS = {'trace_id','component','outcome','duration_ms','name','level','span_id','ts'}
def safe_event(fields):
    return {k: fields[k] for k in SAFE_KEYS if k in fields}
candidate = safe_event({'name':'request', 'trace_id':'tr-2', 'authorization':'synthetic-secret', 'outcome':'ok'})
assert 'authorization' not in candidate
print(candidate)


In [ ]:
# Reference checks: counters, bounded latency, and correlation.
api_request('tr-2', 0.01, fail=True)
worker('tr-2')
assert tel.counters['requests.total'] == 2
assert tel.counters['requests.error'] == 1
assert max(tel.latency_ms) == 40.0
assert all('authorization' not in e for e in tel.events)
assert {e['trace_id'] for e in tel.events if e['trace_id'] == 'tr-2'} == {'tr-2'}
print('Telemetry checks passed')


## Diagnose before source
The `tr-1` trace has a store span of 40 ms, while the `tr-2` trace has an error. State the evidence: which component and outcome would you investigate first?


## Intentionally weak AI-style instrumentation
A generated logger might emit the entire request payload and use `user_id` as a metric label. Critique it: raw payloads can contain secrets; user IDs create unbounded cardinality. The bounded event policy above avoids both.


## Independent challenge — attempt before checking
Define an alert with a symptom, threshold, window, owner, and action. Write your alert in notes first, then compare with the executable shape below.


In [ ]:
alert = {'metric':'latency_ms.p95','threshold_ms':30,'windows':3,'owner':'reports-owner','action':'inspect store.read and dependency health'}
assert all(key in alert for key in ('threshold_ms','windows','owner','action'))
print(alert)


## Exit questions
1. What are logs, metrics, and traces each best at showing?
2. What does readiness prove?
3. What remains unproved?

### Answers
1. Event detail, aggregate rates/distributions, and one end-to-end journey.
2. The documented workflow can serve, not that every dependency is globally perfect.
3. Vendor transport, retention, sampling, and production cardinality.

## Evidence handoff
Save redacted events, counters/latency, correlated timeline, health results, diagnosis note, alert owner/action, and AI diff critique.
